# Chatterbox v3: three ways to render a group of segments

The pipeline currently joins text into one prompt and calls `ChatterboxTTS.generate`
once. That is where the audio cut-offs come from: upstream hardcodes
`max_new_tokens=1000`, and at 25 speech tokens per second that is a hard 40 second
ceiling, after which the last sentence stops mid word.

This notebook renders the same segments three ways so the difference can be judged
by ear before the app commits to an architecture.

| Variant | What it does | Why it is here |
|---|---|---|
| `render_stock` | joined prompt, unmodified upstream sampler | the control, exactly today's behavior |
| `render_joined` | joined prompt, our sampler with the token cap exposed | shows whether the cut-off is only the cap |
| `render_tensor_batch` | segments padded into one batch, one waveform each | every segment gets the full budget, one transformer visit per decode step |

## Prerequisites

- A kernel where `uv pip install --editable ".[chatterbox]"` has run.
- Weights download into `models/chatterbox` on first use, never the HF cache.
- CPU and CUDA both work; CUDA is only faster.
- A reference wav is optional. Without one the builtin voice in `conds.pt` is used.

In [ ]:
text_sample = """
The challenge for us is that no amount of studying or open-mindedness can
genuinely recreate the power of fear and uncertainty
I can read about what it was like to lose everything during the Great Depression. But
I don’t have the emotional scars of those who actually experienced it. And the person
who lived through it can’t fathom why someone like me could come across as
complacent about things like owning stocks. We see the world through a different
lens.
Spreadsheets can model the historic frequency of big stock market declines. But
they can’t model the feeling of coming home, looking at your kids, and wondering if
you’ve made a mistake that will impact their lives. Studying history makes you feel
like you understand something. But until you’ve lived through it and personally felt
its consequences, you may not understand it enough to change your behavior.
We all think we know how the world works. But we’ve all only experienced a tiny
sliver of it.


Dogs were domesticated 10,000 years ago and still retain some behaviors of their
wild ancestors. Yet here we are, with between 20 and 50 years of experience in the
modern financial system, hoping to be perfectly acclimated.
For a topic that is so influenced by emotion versus fact, this is a problem. And it
helps explain why we don’t always do what we’re supposed to with money.
We all do crazy stuff with money, because we’re all relatively new to this game and
what looks crazy to you might make sense to me. But no one is crazy—we all make
decisions based on our own unique experiences that seem to make sense to us in a
given moment.

Modern capitalism is a pro at two things: generating wealth and generating envy.
Perhaps they go hand in hand; wanting to surpass your peers can be the fuel of hard
work. But life isn’t any fun without a sense of enough. Happiness, as it’s said, is just
results minus expectations.

The only way to know how much food you can eat is to eat until you’re sick. Few try
this because vomiting hurts more than any meal is good. For some reason the same
logic doesn’t translate to business and investing, and many will only stop reaching
for more when they break and are forced to. This can be as innocent as burning out
at work or a risky investment allocation you can’t maintain. On the other end there’s
Rajat Guptas and Bernie Madoffs in the world, who resort to stealing because every
dollar is worth reaching for regardless of consequence.

There were plenty of theories about why ice ages occurred. To account for their
enormous geological influence the theories were equally grand. The uplifting of
mountain ranges, it was thought, may have shifted the Earth’s winds enough to alter
the climate. Others favored the idea that ice was the natural state, interrupted by
massive volcanic eruptions that warmed us up. But none of these theories could account for the cycle of ice ages. The growth of
mountain ranges or some massive volcano may explain one ice age. It could not
explain the cyclical repetition of five.
"""

In [2]:
import re

def clean_corpus(text: str) -> str:
    # Normalize curly/smart quotes.
    text = (
        text.replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
    )
    # Remove digits immediately following a period: "word.123" -> "word."
    text = re.sub(r"\.(\d+)", ".", text)
    # Preserve paragraph breaks (2+ newlines), but unwrap single line breaks.
    parts = re.split(r"(\n{2,})", text)
    for index in range(0, len(parts), 2):
        parts[index] = re.sub(r"[ \t]*\n[ \t]*", " ", parts[index])
    return "".join(parts)


text_sample = """
The astrophysicist, however, does not make the missiles or the bombs.
Astrophysicists make no weapons at all. Instead, we and the military happen to care
about many of the same things: multi-spectral detection, ranging, tracking, imaging,
high ground, nuclear fusion, access to space. 
Looking down at Earth from an orbiting spacecraft, a rational person could certainly
feel that "necessary defence" may have more to do with the vulnerability of our
beautiful blue planet, exposed to all the vicissitudes of the cosmos, than with the
transient power of a single country’s weapons, policymakers, nationalists, and
ideologues, however virulent. From hundreds of kilometers above the surface of the
globe, "Peace on Earth, Goodwill Toward Men" might sound less like a standard line
on a Christmas card and more like an essential step toward a viable future, in which
all of humankind cooperates in protecting Earth from the enemies among us and the
threats above us.
But none of those expressions of national pride and militarism meshed with my
sense of armed conflict. I didn’t understand how they fit together. That
twenty-something grease monkey did, though. He was plugged into a primal passion
that has energized so many wars across the millennia. Just not the war I’d grown up
with.
With the winding-down of the Cold War, the aerospace industry had undergone
unrelenting shrinkage and consolidation. Seventy-five aerospace companies were in
operation on the day Reagan was elected, merging into sixty-one by the time the
Berlin Wall fell, and finally into just five titans—Lockheed Martin, Boeing, Raytheon,
Northrop Grumman, and General Dynamics—by the time the Twin Towers crumbled
into toxic dust. Some 600,000 scientific and technical jobs had vanished in just a
dozen of those years, along with incalculable quantities of experience and
intellectual capital.
Gone are the days when a nation’s security could be measured by bombers and
battleships. Security in this new era requires harnessing all of America’s strengths:
the force of our diplomacy; the might of our military; the vigor and competitiveness
of our economy; and the power of our ideals.
Has the United States funded and engaged in this kind of research purely for the
sake of discovery? Hardly. Most American accelerators were built during the Cold
War, when the particle physicist was a vital resource for increasing the lethality of
nuclear weapons. That’s how astrophysics—specifically astro-particle physics, a
branch of cosmology—became an auxiliary beneficiary of Cold War science priorities.
Astrophysics and the military are conjoined whether their shared buoy bobs up or
down in the tidewaters of politics.
Consider units of time. If the Sun never set and the Moon never waned, our measures
of time might be grounded solely in biology—the beating heart, circadian rhythms,
menstruation—because “periodicity is part of who we are.”1 But the Sun does set,
and the Moon predictably waxes and wanes. Transitions recur endlessly in the skies
above. Celestial cycles offer themselves as a natural measure of time in units we
care about.
The modern distinction between astronomy and astrology was, at one time, fuzzy
and irrelevant, just like the distinction between alchemy and chemistry or between
magic and medicine. Promoting the propitious, avoiding calamity, and forecasting
death suggest astrological interpretation. Yet prediction, which is an offshoot of
analysis, can be exacting and scientific, given the right practitioner. Accurate
observation of the skies, combined with a grasp of physics and a cartography of the
cosmos, is the cornerstone of both.
The full electromagnetic spectrum may extend forever in both directions: toward
ever longer wavelengths, perhaps limited by the size of the universe itself, and
toward ever smaller wavelengths, perhaps limited by quantum physics. Currently we
have the technology to detect wavelengths ranging from less than a
hundred-billionth of a meter (high-frequency gamma rays) to many hundreds of
kilometers (extremely low-frequency radio waves). That’s a factor of quadrillions.
Millennia ago, if you wanted to look up at the sky or across a broad valley, you might
have used a long sighting tube to focus your attention and reduce glare, as did
Aristotle and probably his predecessors. But no empty tube, however long—and
whether cast in gold by an ancient Assyrian metalworker, carved from jade by an
ancient Chinese artisan, or fastened to an armillary sphere by a mathematically
astute medieval pope1—would improve your physiological capacity to detect the
planet Neptune or assess the size of a rival army or navy massing on a distant shore.
Put a couple of lenses in the tube, though, and you’ve got yourself an optical
telescope.
A tool for augmenting the senses, the telescope enables you both to detect things
too faint to see and to resolve detail where your eyes would otherwise fail you. First,
it shows you that an object exists; then, by revealing the object’s shape, motion, and
color, it hints at what the object might be. The telescope’s task is to collect at a
distance as much visual information as it can, and feed it to your brain via your eyes.

Fortunately for astrophysicists, the germ of a better solution was already available.
In 1835 the German chemist Justus von Liebig had introduced the silvered-glass
mirror. Made by depositing a thin layer of silver vapor on one side of a slab of
polished glass, it offered an excellent image and soon became a fixture of every
bourgeois household. Two decades later, Jean-Bernard-Léon Foucault (the pendulum
fellow), in collaboration with the Paris Observatory’s official optician, improved upon
this technique by adding a subsequent phase: localized repolishing to correct errors
of form. This enabled Foucault to make ever-larger reflecting telescopes, culminating
in an eighty-centimeter telescope installed in the Marseille Observatory in 1864.71
Today the largest telescopes in the world are all reflectors, and all of them use a
mirror with a vapor-deposited metal coating on one polished glass surface. While the
lens of the largest extant refracting telescope is one meter across, the mirror of the
largest reflecting telescope is more than ten meters across. Others in the works
approach forty meters in diameter. Hardly anything limits the size of the mirror,
because it is mounted from the back. As a result, since the end of the nineteenth
century, the reflector has been the astrophysicist’s instrument of choice.


The military solution, however, lay elsewhere. For nearly the entire nineteenth
century, military planners and artillerymen alike fretted far less than astronomers
about the limited availability of fine optical glass. An infantry rifle that could be fired
effectively at a target more than a mile away was not yet on the market.72 Gunners
did not rely on barrel-mounted spotting scopes. Civil War cannons were fired point
blank in the general direction of a nearby visible enemy; battling Northerners and
Southerners estimated distances strictly by eye and aimed their guns with the aid of
spirit levels and plumb lines, hoping to overwhelm the enemy with a barrage of shot.
“The gunners sighted their fieldpieces hastily and banged away, trusting to hit some
vital spot,” writes Lieutenant Colonel F. E. Wright in a historical overview produced in
1921 for the Ordnance Department of the US Army.
The modern, Western offspring of astronomy, astrophysics is not even a century and
a half old. Its midwives were two nineteenth-century technological innovations. The
more widely known of the two, photography—literally “light drawing”—stemmed
from a welter of investigations into the image-forming proclivities of light. The
lesser-known and more specialized innovation, spectroscopy—which separates light
into its component colors, yielding heaps of information about its source—derived
from the prismatic study of the Sun’s spectrum and the discovery that every
substance radiates a characteristic and unique combination of colors. Jointly,
photography and spectroscopy empowered the astronomer to record and analyze
whatever light the available telescopes could gather from the sky.
Creating confusion seemed a better solution than trying to attain invisibility.
Suddenly artists became facilitators of military goals, as warriors picked up some of
the disintegrationist, scientistic visual strategies of the vanguard movements of
Cubism, Futurism, and Vorticism. Picasso and Braque, the fathers of Cubism, were
delighted to see what they regarded as their aesthetic invention being applied to
ships and weaponry; walking down a boulevard in Paris one evening and seeing a
convoy of zigzag-painted heavy guns heading for the front, Picasso reportedly
exclaimed, “We invented that!” Franklin D. Roosevelt himself, assistant secretary of
the US Navy during World War I, is said to have shouted, after being shown a
bedazzled test ship, “How the hell do you expect me to estimate the course of a
God-damn thing all painted up like that?” In the end, though, standardized dazzle
camouflage apparently did not live up to its promise. Attacks proceeded at similar
rates on ships with and without such a coating. Nevertheless, despite plenty of
evidence to the contrary, magical thinking about the efficacy of disruptive
camouflage persisted through World War II and beyond.
Several options for disappearing from the visible part of the spectrum have a long
wartime pedigree. The simplest is to exploit the darkness of night. Another is to blind
the enemy. Set a huge bonfire, and enemy forces who look at it will be unable to see
anything other than flames and therefore be unable to target you with any precision.
In recent decades, both lasers and smokescreens have been used to blind the
enemy: throw a white-phosphorus grenade at your target, and you’ll get an instant
smokescreen that will also scorch whoever is in the vicinity while it masks your own
maneuvers and your own infrared radiation. Blinding also happens to us in the
universe, when the light of a host star swamps the much fainter reflected light of its
exoplanets. This was a big problem until a couple of decades ago, when space
scientists began using a special occulting disk in their telescope’s optics to block out
the offending starlight, thus achieving the opposite of what a telescope was invented
to accomplish.
Another, very different approach to disappearance is transparency—embodied in the
clear glass window. Flies, moths, birds, and visitors from outer space who don’t know
about windows must be baffled by the interposition of something visually
imperceptible but impenetrable between themselves and the scenery.



Objective truths of science are not founded in belief systems. They are not established by the authority of leaders or the power of persuasion. Nor are they learned from repetition or gleaned from magical thinking. To deny objective truths is to be scientifically illiterate, not to be ideologically principled.

Differences in opinion enrich the diversity of a nation, and ought to be cherished and respected in any free society, provided everyone remains free to disagree with one another and, most importantly, everyone remains open to rational arguments that could change your mind.

Daily life in 1960 would be unrecognizable to anyone transported from the year 1930. From 1960 to 1990, a Cold War nuclear arms race between the United States and the Soviet Union threatens the survival of civilization. Though begun in the 1950s, the US stockpile of nuclear warheads peaks in the 1960s, with the Soviets’ stockpile peaking in the 1980s.11 The Berlin Wall, erected in 1962, becomes the greatest symbol of Winston Churchill’s “Iron Curtain,” separating Eastern from Western Europe. Yet it’s dismantled by 1989, as peace breaks out in Europe. The commercialization of the transistor allows consumer electronics to miniaturize, transforming audiovisual equipment from heavy, floor-mounted living room furniture to what you carry in your pocket.

When you chart the transmission curve for microwaves through Earth’s atmosphere,
you find a sudden, transparent window between the wavelengths of eighteen and
twenty-one centimeters. At either end of this little band, radio astronomers can
detect distinct emissions from the universe’s ubiquitous hydrogen atoms (H) and
their partners in water, hydroxyl molecules (OH). This band has been dubbed the
water hole—a phrase more commonly invoked for places on Earth where wild
creatures congregate to drink and wallow. We suspect that any aliens who know of
our existence and want to communicate with us might also know the absorptive
effects of water on various wavelengths. So if they’re clever, they might exploit the
dip and try to reach out to us via the microwave frequencies that get through the
water hole.
How about a less pacific result of the astrophysical discovery that water absorbs
microwaves? How difficult would it be to design a nonlethal weapon that targets the
water content of the human body? Three-fifths of our average body mass is water.
Such a weapon could operate on the same principle as the microwave oven.

When the CCD is exposed to light, whether from a nearby street scene or a faraway
galaxy, each of its pixels stores some number of electrons, depending on the
intensity of the light hitting each of the tiny locations on the CCD’s light-responsive
computer chip. The more intense the light, the more electrons get stored—although
if the light is too bright, you will saturate the detector and the excess electrons will
spill over into neighboring pixels, contaminating their data. Double the exposure,
and you get double the number of electrons. The electrons that congregate in each
pixel are then collected from the chip, tabulated, and turned into a single electronic
tile in the mosaic that constitutes the complete image. The more pixels, the more
resolution available to you. Nowadays you can easily download a street scene from
Wikimedia Commons that measures 2592 columns × 1944 rows, which translates
into a grid of more than 5,000,000 pixels—a crisply detailed photo. But that’s
nothing: if you’re not worried about overtaxing your computer, you can download an
image of the Orion Nebula from the HubbleSite Gallery that’s 18,000 × 18,000—a
grid of 324,000,000 pixels, packed to the gills with detail.
"""

print(clean_corpus(text_sample))

 The astrophysicist, however, does not make the missiles or the bombs. Astrophysicists make no weapons at all. Instead, we and the military happen to care about many of the same things: multi-spectral detection, ranging, tracking, imaging, high ground, nuclear fusion, access to space. Looking down at Earth from an orbiting spacecraft, a rational person could certainly feel that "necessary defence" may have more to do with the vulnerability of our beautiful blue planet, exposed to all the vicissitudes of the cosmos, than with the transient power of a single country's weapons, policymakers, nationalists, and ideologues, however virulent. From hundreds of kilometers above the surface of the globe, "Peace on Earth, Goodwill Toward Men" might sound less like a standard line on a Christmas card and more like an essential step toward a viable future, in which all of humankind cooperates in protecting Earth from the enemies among us and the threats above us. But none of those expressions of na

In [ ]:
import json
import sys
from dataclasses import dataclass, replace
from pathlib import Path

import torch
import torchaudio
from IPython.display import Audio, display

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from lib.inference.chatterbox_batch import (
    SamplingParams,
    group_sentences,
    render_joined,
    render_stock,
    render_tensor_batch
)
from lib.conf import max_vram_gb
from lib.inference.model_store import ensure_chatterbox
from lib.modules.text_processing import BatchConfig, SmartSegmenter

print("repo", REPO_ROOT)
print("cuda", torch.cuda.is_available())

## Run configuration

`mtl_version` picks the T3 shard: `v2` loads `t3_mtl23ls_v2.safetensors`, `v3` loads
`t3_mtl23ls_v3.safetensors` from the same model card. Everything else on disk is shared,
so switching versions costs one extra file, not a second checkout.

In [ ]:
@dataclass
class RunConfig:
    model_type: str = "multilingual"       # english or multilingual
    mtl_version: str = "v3"                # v2 or v3, multilingual only
    language_id: str = "en"                # one of the 23 supported codes
    reference_audio: str | None = None     # path to a 5-20s wav, or None for the builtin voice
    per_batch: int = 8                     # segments per batch, the app default
    max_tokens_per_batch: int = 48         # segmenter budget, matches the pipeline
    max_new_tokens: int = 1000             # upstream cap, 40 seconds of audio
    exaggeration: float = 0.5
    cfg_weight: float = 0.5
    temperature: float = 0.8
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir: Path = REPO_ROOT / "tmp" / "notebook_out" / "chatterbox_v3"


cfg = RunConfig()
cfg.out_dir.mkdir(parents=True, exist_ok=True)
cfg

## VRAM ceiling

The app caps every CUDA job before the model is loaded, so a batch that is too wide fails
inside this job instead of taking down whatever else shares the card. The notebook does the
same thing, with the same default, so the memory numbers below are the numbers the app would
see. Lower `vram_cap_gb` to find the widest batch that still fits your budget.

In [ ]:
vram_cap_gb = max_vram_gb

def apply_vram_cap(device: str, cap_gb: int) -> None:
    """Mirror the app's per job ceiling, must run before any weights are loaded."""
    if not device.startswith("cuda"):
        print("no cap, device is", device)
        return
    index = int(device.split(":")[1]) if ":" in device else 0
    total_gb = torch.cuda.get_device_properties(index).total_memory / 1024 ** 3
    if total_gb <= cap_gb:
        print(f"no cap, card is {total_gb:.1f}GB and the ceiling is {cap_gb}GB")
        return
    torch.cuda.set_per_process_memory_fraction(min(1.0, cap_gb / total_gb), index)
    print(f"capped to {cap_gb}GB of {total_gb:.1f}GB on cuda:{index}")


def reset_peak() -> None:
    if cfg.device.startswith("cuda"):
        torch.cuda.reset_peak_memory_stats()


def peak_gb() -> float:
    """Peak allocated VRAM since the last reset, in GB, zero off CUDA."""
    if not cfg.device.startswith("cuda"):
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 3


apply_vram_cap(cfg.device, vram_cap_gb)

## Load the model

`ensure_chatterbox` downloads only the shards this configuration needs into
`models/chatterbox` and returns the directory; `from_local` never reaches the network.

In [ ]:
def load_model(cfg: RunConfig) -> object:
    """Load Chatterbox from models/chatterbox, fetching missing shards first."""
    ckpt_dir = ensure_chatterbox("", cfg.model_type, cfg.mtl_version)
    if cfg.model_type == "english":
        from chatterbox.tts import ChatterboxTTS
        model = ChatterboxTTS.from_local(ckpt_dir=ckpt_dir, device=cfg.device)
    else:
        from chatterbox.mtl_tts import ChatterboxMultilingualTTS
        model = ChatterboxMultilingualTTS.from_local(
            ckpt_dir=ckpt_dir, device=cfg.device, t3_model=cfg.mtl_version
        )
    if cfg.reference_audio:
        model.prepare_conditionals(cfg.reference_audio, exaggeration=cfg.exaggeration)
    return model


reset_peak()
model = load_model(cfg)
print(type(model).__name__, "sample rate", model.sr, "on", cfg.device)
print(f"weights peak {peak_gb():.2f}GB")
weights_gb = peak_gb()

## Segment the text

The app's own `SmartSegmenter` is used rather than a notebook-local splitter, so what is
measured here is what the pipeline would feed the engine. Each segment is a token-budgeted
group of whole sentences; `group_sentences` then packs `per_batch` of them into one batch.

In [ ]:
SAMPLE_TEXT = """
The lighthouse keeper had not spoken to another person in eleven weeks.
He kept a log, as the service required, and the log was mostly weather.
Wind from the south-west, thirty knots. Rain after midnight. Nothing sighted.

On the twelfth week a boat came, and it came at the wrong hour, riding the swell
without lights. He watched it from the gallery with the lamp turning behind him,
throwing his shadow out over the water in long slow strokes.

The man who climbed the ladder was younger than expected, and soaked through, and
he carried a leather case that he would not put down. He said he had come about
the records. He said it twice, as though the repetition settled the matter.

The keeper made tea because that was what one did, and because it bought him time
to think about which records a man would cross nine miles of open water for.
"""

segmenter = SmartSegmenter(BatchConfig(max_tokens_per_batch=cfg.max_tokens_per_batch))
result = segmenter.segment_text(SAMPLE_TEXT)
segments = [seg.text for seg in result.segments]
groups = group_sentences(segments, per_batch=cfg.per_batch)

print(result.preview())
print("\ngroups:", [len(g) for g in groups])

## Render all three variants

`stock` and `joined` produce one waveform for the whole group; `tensor` produces one per
segment. Timings for the tensor variant are the shared sampling cost divided across rows,
which is the fair per-segment number.

In [ ]:
params = SamplingParams(
    exaggeration=cfg.exaggeration,
    cfg_weight=cfg.cfg_weight,
    temperature=cfg.temperature,
    max_new_tokens=cfg.max_new_tokens,
    language_id=cfg.language_id if cfg.model_type != "english" else None
)


def record(variant: str, index: int, rendered: object, peak: float = 0.0) -> dict:
    """Write one waveform to out_dir and return its metadata row."""
    name = f"{variant}_{index:02d}.wav"
    torchaudio.save(str(cfg.out_dir / name), rendered.wav.cpu(), model.sr)
    return {
        "variant": variant,
        "index": index,
        "file": name,
        "chars": len(rendered.text),
        "speech_tokens": rendered.speech_tokens,
        "duration_sec": round(rendered.duration_sec, 2),
        "inference_sec": round(rendered.inference_sec, 2),
        "rtf": round(rendered.inference_sec / max(rendered.duration_sec, 1e-6), 3),
        "truncated": rendered.truncated,
        "peak_vram_gb": round(peak, 2),
        "over_weights_gb": round(max(peak - weights_gb, 0.0), 2),
        "text": rendered.text
    }


group = groups[0]
rows = []
for variant, render in (("stock", render_stock), ("joined", render_joined)):
    reset_peak()
    rows.append(record(variant, 0, render(model, group, params), peak_gb()))

reset_peak()
tensor_rendered = render_tensor_batch(model, group, params)
tensor_peak = peak_gb()
rows += [record("tensor", i, r, tensor_peak) for i, r in enumerate(tensor_rendered)]

(cfg.out_dir / "generation_metadata.json").write_text(json.dumps(rows, indent=2))
for row in rows:
    print(f"{row['variant']:7s} {row['index']:02d} "
          f"{row['duration_sec']:7.2f}s tokens={row['speech_tokens']:5d} "
          f"rtf={row['rtf']:6.2f} truncated={row['truncated']} "
          f"peak={row['peak_vram_gb']:5.2f}GB chars={row['chars']}")

### Totals

The number that matters for the app is wall clock for the whole group and whether anything
was cut. A tensor batch that is slower per step can still win, because it pays the
transformer once for the whole batch instead of once per segment.

In [ ]:
for variant in ("stock", "joined", "tensor"):
    part = [r for r in rows if r["variant"] == variant]
    if not part:
        continue
    audio = sum(r["duration_sec"] for r in part)
    wall = sum(r["inference_sec"] for r in part)
    cut = sum(1 for r in part if r["truncated"])
    peak = max(r["peak_vram_gb"] for r in part)
    print(f"{variant:7s} audio={audio:7.2f}s wall={wall:7.2f}s "
          f"rtf={wall / max(audio, 1e-6):5.2f} truncated_rows={cut}/{len(part)} "
          f"peak={peak:5.2f}GB (+{max(peak - weights_gb, 0.0):4.2f}GB over weights)")

## Listen

Play the two joined variants first, then the stitched tensor batch. What to listen for is
listed at the bottom of the notebook.

In [ ]:
def stitch(files: list[Path], gap_ms: int = 120) -> torch.Tensor:
    """Concatenate waveforms with a short silence, the way the pipeline stitches chunks."""
    gap = torch.zeros(1, int(model.sr * gap_ms / 1000))
    parts = []
    for path in files:
        wav, _ = torchaudio.load(str(path))
        parts.extend([wav, gap])
    return torch.cat(parts[:-1], dim=-1)


tensor_files = [cfg.out_dir / r["file"] for r in rows if r["variant"] == "tensor"]
tensor_joined = stitch(tensor_files)
torchaudio.save(str(cfg.out_dir / "tensor_stitched.wav"), tensor_joined, model.sr)

for label, path in (
    ("stock (today's behavior)", cfg.out_dir / "stock_00.wav"),
    ("joined (our sampler)", cfg.out_dir / "joined_00.wav"),
    ("tensor batch, stitched", cfg.out_dir / "tensor_stitched.wav")
):
    print(label)
    display(Audio(str(path)))

In [ ]:
for row in [r for r in rows if r["variant"] == "tensor"]:
    print(f"[{row['index']:02d}] {row['duration_sec']:.2f}s  {row['text'][:90]}")
    display(Audio(str(cfg.out_dir / row["file"])))

## Where the cut-off comes from

`T3Config.max_speech_tokens` is 4096, so the 1000 token cap in `generate` is a choice, not a
limit of the model. Sweeping it on the joined variant shows the cliff directly: below the cap
the duration keeps growing, at the cap `truncated` flips and the tail is lost.

In [ ]:
for cap in (500, 1000, 2000, 4096):
    rendered = render_joined(model, group, replace(params, max_new_tokens=cap))
    torchaudio.save(str(cfg.out_dir / f"cap_{cap}.wav"), rendered.wav.cpu(), model.sr)
    print(f"cap={cap:5d} tokens={rendered.speech_tokens:5d} "
          f"audio={rendered.duration_sec:7.2f}s truncated={rendered.truncated}")

In [ ]:
for cap in (500, 1000, 2000, 4096):
    print("cap", cap)
    display(Audio(str(cfg.out_dir / f"cap_{cap}.wav")))

## Cross lingual cloning

v3 is the multilingual checkpoint: 23 languages, zero-shot cloning from a reference clip, and
the reference does not have to be in the target language. The model card warns about accent
bleed from the reference and recommends `cfg_weight=0` for cross lingual work, which is what
the cell below uses.

Run this with `cfg.reference_audio` pointing at an English clip and listen for how much English
accent survives into the other languages.

In [ ]:
CROSS_LINGUAL = {
    "en": "The keeper made tea, because that was what one did on a night like this.",
    "fr": "Le gardien prepara du the, car c'est ce que l'on fait par une nuit pareille.",
    "de": "Der Waerter kochte Tee, denn das tat man in einer solchen Nacht.",
    "es": "El farero preparo te, porque eso es lo que se hace en una noche asi.",
    "it": "Il guardiano prepara il te, perche e cio che si fa in una notte simile.",
    "ja": "灯台守はお茶をいれた。こんな夜にはそうするものだからだ。",
    "zh": "守塔人泡了茶，因为这样的夜晚本该如此。"
}

if cfg.model_type == "english":
    print("english checkpoint has no language_id, reload with model_type='multilingual'")
else:
    cross_params = replace(params, cfg_weight=0.0, max_new_tokens=2000)
    cross_rows = []
    for lang, text in CROSS_LINGUAL.items():
        rendered = render_joined(model, [text], replace(cross_params, language_id=lang))
        torchaudio.save(str(cfg.out_dir / f"cross_{lang}.wav"), rendered.wav.cpu(), model.sr)
        rtf = round(rendered.inference_sec / max(rendered.duration_sec, 1e-6), 3)
        cross_rows.append({
            "lang": lang,
            "text": text,
            "duration_sec": round(rendered.duration_sec, 2),
            "rtf": rtf
        })
        print(f"{lang}  {rendered.duration_sec:6.2f}s  rtf={rtf:5.2f}")
    (cfg.out_dir / "cross_lingual_metadata.json").write_text(json.dumps(cross_rows, indent=2))

In [ ]:
if cfg.model_type != "english":
    for lang in CROSS_LINGUAL:
        print(lang, CROSS_LINGUAL[lang][:60])
        display(Audio(str(cfg.out_dir / f"cross_{lang}.wav")))

## What to judge

1. **Cut-offs.** Does `stock` lose the end of the group, and does raising the cap on `joined`
   fix it, or does quality fall apart before the audio does?
2. **Prosody across boundaries.** The joined variants hear the whole group as one utterance, so
   sentence transitions should flow. The tensor variant renders segments independently, which
   can sound clipped or over-reset at each boundary.
3. **Speaker drift.** Compare the first and last tensor rows against each other. Independent
   sampling from the same conditioning can drift in pitch or pace.
4. **Throughput.** Compare the totals cell. Tensor batching wins when the per-step transformer
   cost dominates, which is the CUDA case; on CPU the gap is smaller.
5. **Memory.** `peak_vram_gb` minus the weights baseline is what batching actually costs. A
   tensor batch that beats the ceiling in the cell above is not a candidate, whatever its RTF.
6. **Cross lingual.** How much of the reference accent leaks, and whether `cfg_weight=0` is
   enough to keep it out.

Nothing here is wired into the pipeline. The point is to pick one of these shapes first.